<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW2/HW2_LanguageProcessing_RNN_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [180]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import requests

from torch import nn
from torch import functional as F
from torch import optim

!pip install torchinfo #Apparently this is needed if using longs instead of floats

import matplotlib.pyplot as plt


from torchinfo import summary

In [181]:
torch.manual_seed(1)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


#hyperparameters
lr = 1e-3
epochs = 20
sequence_length = 20
hidden_size = 256

device

device(type='cuda', index=0)

In [182]:
#Imported directly from Dr. Tabkhi's Github
# Step 1: Download the dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text  # This is the entire text data

# Create a character mapping to integers
chars = sorted(list(set(text)))
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode the text into integers
encoded_text = [char_to_int[ch] for ch in text]

# Create sequences and targets
sequences = []
targets = []
for i in range(0, len(encoded_text) - sequence_length):
    seq = encoded_text[i:i+sequence_length]
    target = encoded_text[i+sequence_length]
    sequences.append(seq)
    targets.append(target)

# Convert lists to PyTorch tensors
sequences = torch.tensor(sequences, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)

# Step 3: Create a dataset class
class CharDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        return self.sequences[index], self.targets[index]

# Instantiate the dataset
dataset = CharDataset(sequences, targets)

# Step 4: Create data loaders
batch_size = 1024
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size, pin_memory=True)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size, pin_memory=True)

if(train_loader and test_loader):
  print("Train and Test Loader Created")
else:
  print("Not created")

# Now `train_loader` and `test_loader` are ready to be used in a training loop

Train and Test Loader Created


In [183]:
#Create Model
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(RNN, self).__init__()
    self.embedding = nn.Embedding(input_size, hidden_size)
    self.rnn = nn.LSTM(
        input_size = hidden_size,
        hidden_size=hidden_size,
        num_layers=3,
        batch_first = True
    )
    self.fc1 = nn.Linear(in_features = hidden_size, out_features = output_size)


  def forward(self, x):
    embedded = self.embedding(x)
    output,_ = self.rnn(embedded)
    output = self.fc1(output[:, -1, :])
    return output

In [184]:
model = RNN(batch_size, hidden_size, batch_size).to(device)
criterion = nn.CrossEntropyLoss()
optim = optim.Adam(model.parameters(), lr = lr)

In [185]:
def createPlot(sequence_length, train_loss_list, val_loss_list, val_accuracy_list):
  plt.figure(figsize=(12, 5))

  # Plot Loss
  plt.subplot(1, 2, 1)
  plt.plot(train_loss_list, label='Train Loss')
  plt.plot(val_loss_list, label='Val Loss')
  plt.title(f'Loss Curves Sequence Length:{sequence_length}')
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.legend()

  # Plot Accuracy
  plt.subplot(1, 2, 2)
  plt.plot(val_accuracy_list, label='Val Accuracy')
  plt.title(f'Validation Accuracy Sequence Length:{sequence_length}')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy (%)')
  plt.legend()

  plt.show()

In [186]:
###This is where training begins
#Lists for storing loss and validation values
train_loss_list = []
val_loss_list = []
val_accuracy_list = []


#Create a new training loop for each input_length
for epoch in range(epochs):
  model.train()
  for data in train_loader:
    X_train, y_train = data
    X_train = X_train.to(device)
    y_train = y_train.to(device)
    optim.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optim.step()  # Update model parameters




    #Here is where we evaluate the model on the current epoch
  model.eval()
  with torch.no_grad():
    for data in test_loader:
      X_val, y_val = data
      X_val = X_val.to(device)
      y_val = y_val.to(device)
      val_output = model(X_val) # Take test dataset and run it through this epoch's model

      val_loss = criterion(val_output, y_val) #Find the loss

      _, predicted = torch.max(val_output, 1) #Here we find what the output was (what letter)

      val_accuracy = (predicted == y_val).float().mean() #Here we take each answer from out model,
                                                          #compare it to the ground truth, and find how accurate we are

  val_loss_list.append(val_loss.item())
  val_accuracy_list.append(val_accuracy.item())
  train_loss_list.append(loss.item()) #Take this epoch's training loss and add it
                                        #to the training loss list (of all epochs)

  print(f'Epoch {epoch}, Loss: {loss.item():.4f}, Val Accuracy: {val_accuracy.item():.4f}, Val Loss: {val_loss.item():.4f}')

createPlot(sequence_length, train_loss_list, val_loss_list, val_accuracy_list)



Epoch 0, Loss: 2.1172, Val Accuracy: 0.4111, Val Loss: 2.0596
Epoch 1, Loss: 1.6459, Val Accuracy: 0.4971, Val Loss: 1.6790
Epoch 2, Loss: 1.8059, Val Accuracy: 0.5482, Val Loss: 1.5238


KeyboardInterrupt: 

In [ ]:
summary(model, input_size = (1,20), dtypes=[torch.long])

In [ ]:
def predict_next_char(model, char_to_ix, ix_to_char, initial_str):
    model.eval()
    temperature = 0.8
    with torch.no_grad():
        initial_input = torch.tensor([char_to_ix[c] for c in initial_str[-sequence_length:]], dtype=torch.long).unsqueeze(0).to(device)
        prediction = model(initial_input)
        probabilities = prediction[0] / temperature
        predicted_index = torch.softmax(probabilities, dim=-1)
        predicted_index = torch.multinomial(predicted_index, num_samples=1).item()
        return ix_to_char[predicted_index]

In [ ]:
text = "Hi "
for i in range(1000):
  text += predict_next_char(model, char_to_int, int_to_char, text)
print(text)